In [3]:
from pathlib import Path
import numpy as np

# =========================================================
# Resolve Project Paths
# =========================================================

cwd = Path.cwd().resolve()

if (cwd / "dataset").exists():

    project_root = cwd

elif (cwd.parent / "dataset").exists():

    project_root = cwd.parent

else:

    raise FileNotFoundError(
        "Cannot find project root. Run this notebook from the repo root "
        "or from the 19_pure_tag directory."
    )

experiment_dir = project_root / "19_pure_tag"
experiment_dir.mkdir(exist_ok=True)

legacy_dir = project_root / "11_without_75_tags"

print("Project root:", project_root)

# =========================================================
# Load Matched Indices
# =========================================================

matched_indices_path = legacy_dir / "matched_indices_no_overlap.npy"

if not matched_indices_path.exists():

    matched_indices_path = project_root / "matched_indices.npy"

matched_indices = np.load(
    matched_indices_path
).astype(np.int64)

print("Matched indices:", matched_indices_path)
print("Matched samples:", len(matched_indices))

# =========================================================
# Load Labels
# =========================================================

labels_path = legacy_dir / "database_labels_81_big.npy"

if not labels_path.exists():

    labels_path = project_root / "dataset" / "database_labels_81_big.npy"

labels_all = np.load(
    labels_path,
    mmap_mode="r"
)

labels = labels_all[matched_indices].astype(np.float32)

num_classes = labels.shape[1]

print("\nLabels:", labels_path)
print("Labels shape:", labels.shape)

# =========================================================
# Load CLEAN Tag Feature
# - Prefer experiment-11 no-overlap tags if they exist.
# - Otherwise remove label-overlap columns from aligned_tag_feature.npy.
# =========================================================

tag_no_overlap_path = legacy_dir / "aligned_tag_feature_no_overlap.npy"

def load_rows_and_columns(path, rows, columns=None, chunk_size=8192):

    source = np.load(
        path,
        mmap_mode="r"
    )

    if columns is None:

        return source[rows].astype(np.float32)

    output = np.empty(
        (len(rows), len(columns)),
        dtype=np.float32
    )

    for start in range(0, len(rows), chunk_size):

        end = min(
            start + chunk_size,
            len(rows)
        )

        output[start:end] = source[
            np.ix_(rows[start:end], columns)
        ]

    return output

if tag_no_overlap_path.exists():

    tags = load_rows_and_columns(
        tag_no_overlap_path,
        matched_indices
    )

    removed_tag_count = "precomputed"

else:

    tag_feature_path = project_root / "aligned_tag_feature.npy"
    tag_list_path = project_root / "dataset" / "NUS_WID_Tags" / "TagList1k.txt"
    concept_path = project_root / "Concepts81.txt"

    with open(tag_list_path, "r", encoding="utf-8") as f:

        tag_names = [
            line.strip().lower()
            for line in f
            if line.strip()
        ]

    with open(concept_path, "r", encoding="utf-8") as f:

        label_names = {
            line.strip().lower()
            for line in f
            if line.strip()
        }

    overlap_indices = np.array(
        [
            idx
            for idx, tag in enumerate(tag_names)
            if tag in label_names
        ],
        dtype=np.int64
    )

    overlap_index_set = set(overlap_indices.tolist())

    keep_tag_indices = np.array(
        [
            idx
            for idx in range(len(tag_names))
            if idx not in overlap_index_set
        ],
        dtype=np.int64
    )

    tags = load_rows_and_columns(
        tag_feature_path,
        matched_indices,
        keep_tag_indices
    )

    removed_tag_count = len(overlap_indices)

print("\nRemoved label-overlap tag columns:", removed_tag_count)
print("Clean tag feature shape:", tags.shape)

# =========================================================
# Single View: TAG only
# =========================================================

aligned_views = [tags]

view_dims = [
    view.shape[1]
    for view in aligned_views
]

assert len(labels) == len(tags), (
    f"labels/tags sample mismatch: {len(labels)} vs {len(tags)}"
)

assert num_classes == 81, (
    f"expected 81 classes, got {num_classes}"
)

print("\nTotal views:", len(aligned_views))
print("View dimensions:", view_dims)

print("\n================================================")
print("PURE TAG PIPELINE READY")
print("================================================")


Project root: /home/jue/760
Matched indices: /home/jue/760/matched_indices.npy
Matched samples: 116127

Labels: /home/jue/760/dataset/database_labels_81_big.npy
Labels shape: (116127, 81)

Removed label-overlap tag columns: 75
Clean tag feature shape: (116127, 925)

Total views: 1
View dimensions: [925]

PURE TAG PIPELINE READY


In [4]:
import torch

try:

    from torch.utils.data import Dataset
    from torch.utils.data import DataLoader
    from torch.utils.data import random_split

except ModuleNotFoundError as exc:

    raise ModuleNotFoundError(
        "PyTorch is not available in this kernel. "
        "Use the conda env at /home/jue/miniconda3/envs/ml/bin/python."
    ) from exc

# =========================================================
# Multi-View Dataset
# =========================================================

class MultiViewDataset(Dataset):

    def __init__(self, views, labels):

        self.views = views
        self.labels = labels

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        sample_views = [
            torch.from_numpy(view[idx])
            for view in self.views
        ]

        label = torch.from_numpy(
            self.labels[idx]
        )

        return sample_views, label

# =========================================================
# Build Dataset
# =========================================================

dataset = MultiViewDataset(
    views=aligned_views,
    labels=labels
)

print("Dataset size:", len(dataset))

# =========================================================
# Train / Validation Split
# =========================================================

train_size = int(0.9 * len(dataset))

val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print("\nTrain size:", len(train_dataset))
print("Val size  :", len(val_dataset))

# =========================================================
# DataLoader
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)

print("\nDataLoader built successfully.")

Dataset size: 116127

Train size: 104514
Val size  : 11613

DataLoader built successfully.


In [5]:
import torch
import torch.nn as nn
import numpy as np

# =========================================================
# Feature Encoder
# =========================================================

class FeatureEncoder(nn.Module):

    def __init__(self, input_dim, hidden_dim):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Linear(input_dim, hidden_dim),

            nn.BatchNorm1d(hidden_dim),

            nn.ReLU(),

            nn.Dropout(0.3)
        )

    def forward(self, x):

        return self.encoder(x)

# =========================================================
# Multi-View Backbone
# =========================================================

class MultiViewBackbone(nn.Module):

    def __init__(
        self,
        view_dims,
        num_classes=81
    ):
        super().__init__()

        self.encoders = nn.ModuleList()

        fusion_dim = 0

        # -------------------------------------------------
        # Build Encoder
        # -------------------------------------------------

        for dim in view_dims:

            # semantic tag feature (no-leak version)
            if dim >= 900:

                hidden_dim = 512

            # strong visual feature
            elif dim >= 200:

                hidden_dim = 256

            # lightweight visual feature
            else:

                hidden_dim = 128

            self.encoders.append(
                FeatureEncoder(
                    dim,
                    hidden_dim
                )
            )

            fusion_dim += hidden_dim

        print("\nFusion dimension:", fusion_dim)

        # -------------------------------------------------
        # Fusion MLP
        # -------------------------------------------------

        self.fusion = nn.Sequential(

            nn.Linear(fusion_dim, 1024),

            nn.BatchNorm1d(1024),

            nn.GELU(),

            nn.Dropout(0.3),

            nn.Linear(1024, 512),

            nn.BatchNorm1d(512),

            nn.GELU(),

            nn.Dropout(0.3)
        )

        # -------------------------------------------------
        # Residual Shortcut
        # -------------------------------------------------

        self.shortcut = nn.Linear(
            fusion_dim,
            512
        )

        # -------------------------------------------------
        # Final Classifier
        # -------------------------------------------------

        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, views):

        encoded_views = []

        for encoder, view in zip(
            self.encoders,
            views
        ):

            feat = encoder(view)

            encoded_views.append(feat)

        # -------------------------------------------------
        # Early Fusion
        # -------------------------------------------------

        fused = torch.cat(
            encoded_views,
            dim=1
        )

        # -------------------------------------------------
        # Residual Fusion
        # -------------------------------------------------

        deep_feat = self.fusion(fused)

        shortcut_feat = self.shortcut(fused)

        fused_feat = (
            deep_feat +
            shortcut_feat
        )

        # -------------------------------------------------
        # Classification
        # -------------------------------------------------

        logits = self.classifier(
            fused_feat
        )

        return logits

# =========================================================
# Label Correlation Refiner
# =========================================================

class LabelCorrelationRefiner(nn.Module):

    def __init__(
        self,
        correlation_matrix,
        alpha=0.2
    ):
        super().__init__()

        self.alpha = alpha

        self.register_buffer(
            "M",
            torch.tensor(
                correlation_matrix,
                dtype=torch.float32
            )
        )

    def forward(self, logits):

        correlation_update = torch.matmul(
            logits,
            self.M
        )

        refined_logits = (
            logits +
            self.alpha *
            correlation_update
        )

        return refined_logits

# =========================================================
# Full Model
# =========================================================

class MultiViewModel(nn.Module):

    def __init__(
        self,
        backbone,
        refiner=None,
        use_refiner=True
    ):
        super().__init__()

        self.backbone = backbone

        self.refiner = refiner

        self.use_refiner = use_refiner

    def forward(self, views):

        logits = self.backbone(views)

        if (
            self.use_refiner and
            self.refiner is not None
        ):

            logits = self.refiner(logits)

        return logits

# =========================================================
# Build Model
# =========================================================

print("View dimensions:")
print(view_dims)

# =========================================================
# Load Label Graph
# =========================================================

label_graph_path = project_root / "label_graph_fused.npy"

M = np.load(
    label_graph_path
).astype(np.float32)

assert M.shape == (num_classes, num_classes), (
    f"label graph shape mismatch: {M.shape} vs "
    f"({num_classes}, {num_classes})"
)

print("\nGraph:", label_graph_path)
print("Graph shape:", M.shape)

# =========================================================
# Build Backbone
# =========================================================

backbone = MultiViewBackbone(
    view_dims=view_dims,
    num_classes=num_classes
)

# =========================================================
# Graph Refiner
# =========================================================

refiner = LabelCorrelationRefiner(
    correlation_matrix=M,
    alpha=0.2
)

# =========================================================
# Final Model
# =========================================================

model = MultiViewModel(
    backbone=backbone,
    refiner=refiner,
    use_refiner=True
)

# =========================================================
# Device
# =========================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)

print("\nDevice:", device)

# =========================================================
# One-Batch Forward Smoke Test
# =========================================================

model.eval()

with torch.no_grad():

    smoke_views, smoke_labels = next(iter(train_loader))

    smoke_views = [
        v.float().to(device)
        for v in smoke_views
    ]

    smoke_logits = model(smoke_views)

assert smoke_logits.shape == (smoke_labels.shape[0], num_classes), (
    f"logit shape mismatch: {smoke_logits.shape}"
)

print("Smoke logits shape:", tuple(smoke_logits.shape))

print("\n================================================")
print("PURE TAG MODEL BUILT SUCCESSFULLY")
print("================================================")


View dimensions:
[925]

Graph: /home/jue/760/label_graph_fused.npy
Graph shape: (81, 81)

Fusion dimension: 512

Device: cuda
Smoke logits shape: (128, 81)

PURE TAG MODEL BUILT SUCCESSFULLY


In [6]:
from pathlib import Path
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from sklearn.metrics import average_precision_score
import numpy as np

# =========================================================
# Evaluation Metrics
# =========================================================

def evaluate_metrics(y_true, y_prob, threshold=0.5):

    y_pred = (y_prob > threshold).astype(np.float32)

    # -----------------------------------------------------
    # mAP
    # -----------------------------------------------------

    mAP = average_precision_score(
        y_true,
        y_prob,
        average="macro"
    )

    # -----------------------------------------------------
    # Micro-F1
    # -----------------------------------------------------

    micro_f1 = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    # -----------------------------------------------------
    # Macro-F1
    # -----------------------------------------------------

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    return mAP, micro_f1, macro_f1

# =========================================================
# Train Function
# =========================================================

def train_and_evaluate(
    model,
    train_loader,
    val_loader,
    epochs=20,
    lr=1e-4,
    patience=5,
    device=None
):

    # -----------------------------------------------------
    # Device
    # -----------------------------------------------------

    if device is None:

        device = (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    print(f"\nUsing device: {device}")

    model = model.to(device)

    # -----------------------------------------------------
    # Loss Function
    # -----------------------------------------------------

    criterion = nn.BCEWithLogitsLoss()

    # -----------------------------------------------------
    # Optimizer
    # -----------------------------------------------------

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    # -----------------------------------------------------
    # Training State
    # -----------------------------------------------------

    best_map = -float("inf")

    patience_counter = 0

    best_checkpoint_path = (
        Path(experiment_dir) /
        "best_model_pure_tag.pt"
    )

    # =====================================================
    # Epoch Loop
    # =====================================================

    for epoch in range(epochs):

        # =================================================
        # Train Phase
        # =================================================

        model.train()

        total_loss = 0.0

        for views, labels in train_loader:

            views = [
                v.float().to(device)
                for v in views
            ]

            labels = (
                labels.float()
                .to(device)
            )

            optimizer.zero_grad()

            logits = model(views)

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        avg_loss = (
            total_loss /
            len(train_loader)
        )

        # =================================================
        # Validation Phase
        # =================================================

        model.eval()

        all_probs = []
        all_labels = []

        with torch.no_grad():

            for views, labels in val_loader:

                views = [
                    v.float().to(device)
                    for v in views
                ]

                labels = (
                    labels.float()
                    .to(device)
                )

                logits = model(views)

                probs = torch.sigmoid(logits)

                all_probs.append(
                    probs.cpu().numpy()
                )

                all_labels.append(
                    labels.cpu().numpy()
                )

        all_probs = np.concatenate(
            all_probs,
            axis=0
        )

        all_labels = np.concatenate(
            all_labels,
            axis=0
        )

        # =================================================
        # Metrics
        # =================================================

        val_map, mi_f1, ma_f1 = evaluate_metrics(
            all_labels,
            all_probs
        )

        # =================================================
        # Print Metrics
        # =================================================

        print(
            f"Epoch [{epoch+1:03d}/{epochs}] "
            f"| Loss: {avg_loss:.4f} "
            f"| Val mAP: {val_map:.4f} "
            f"| Mi-F1: {mi_f1:.4f} "
            f"| Ma-F1: {ma_f1:.4f}"
        )

        # =================================================
        # Save Best Model
        # =================================================

        if val_map > best_map:

            best_map = val_map

            patience_counter = 0

            torch.save({

                "epoch": epoch + 1,

                "model_state_dict":
                    model.state_dict(),

                "optimizer_state_dict":
                    optimizer.state_dict(),

                "best_map":
                    best_map,

                "view_dims":
                    view_dims,

                "semantic_dim":
                    view_dims[0],

                "experiment":
                    "pure_tag"

            }, best_checkpoint_path)

            print(
                f"Best PURE TAG model saved "
                f"(mAP={best_map:.4f}) -> "
                f"{best_checkpoint_path}"
            )

        else:

            patience_counter += 1

        # =================================================
        # Early Stopping
        # =================================================

        if patience_counter >= patience:

            print("\nEarly stopping triggered.")

            break

    # =====================================================
    # Load Best Model
    # =====================================================

    checkpoint = torch.load(
        best_checkpoint_path,
        map_location=device,
        weights_only=False
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    print("\n================================================")
    print("TRAINING FINISHED")
    print("================================================")

    print(
        f"\nBest Validation mAP: "
        f"{checkpoint['best_map']:.4f}"
    )

    print(
        f"Best epoch: "
        f"{checkpoint['epoch']}"
    )

    return model


In [7]:
trained_model = train_and_evaluate(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=80,
    lr=1e-4,
    patience=25
)


Using device: cuda


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch [001/80] | Loss: 0.1014 | Val mAP: 0.2497 | Mi-F1: 0.4863 | Ma-F1: 0.1362
Best PURE TAG model saved (mAP=0.2497) -> /home/jue/760/19_pure_tag/best_model_pure_tag.pt


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch [002/80] | Loss: 0.0703 | Val mAP: 0.2937 | Mi-F1: 0.5121 | Ma-F1: 0.1983
Best PURE TAG model saved (mAP=0.2937) -> /home/jue/760/19_pure_tag/best_model_pure_tag.pt


...

/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch [042/80] | Loss: 0.0502 | Val mAP: 0.3558 | Mi-F1: 0.5572 | Ma-F1: 0.3279


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Epoch [043/80] | Loss: 0.0500 | Val mAP: 0.3545 | Mi-F1: 0.5604 | Ma-F1: 0.3352
Epoch [044/80] | Loss: 0.0498 | Val mAP: 0.3559 | Mi-F1: 0.5575 | Ma-F1: 0.3300

Early stopping triggered.

TRAINING FINISHED

Best Validation mAP: 0.3604
Best epoch: 19


/home/jue/miniconda3/envs/ml/lib/python3.11/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
